# Deepfake Detection — Full Experiment Pipeline

**Google Colab notebook for training and evaluating XceptionNet and EfficientNet-B0.**

## What this notebook does:
1. Mounts Google Drive (your dataset structure)
2. Sets up Google Drive API for uploading results to shared folder
3. Installs all dependencies
4. Extracts frames from raw videos (FF++ and Celeb-DF)
5. Detects and crops faces using MTCNN
6. Splits data into train/val/test and creates metadata CSVs
7. Trains XceptionNet and EfficientNet-B0
8. Runs all 4 thesis contribution experiments:
   - **C1:** Compression impact quantification (c0/c23/c40)
   - **C2:** Cross-dataset generalization (FF++ <-> Celeb-DF)
   - **C3:** Head-to-head model comparison
   - **C4:** Practical deployment assessment (speed, memory, model size)
9. Generates thesis-quality visualizations
10. Exports metrics for Chapter 4
11. Uploads results to your shared Google Drive folder

## Your Google Drive structure:
```
deepfake_datasets/                    (Datasets account - mounted for reading)
├── faceforensic++/
│   └── FF++/
│       ├── real/
│       └── fake/
└── celeb-df/
    ├── Celeb-real/
    ├── Celeb-synthesis/
    ├── YouTube-real/
    └── list_of_testing_videos.txt

deepfake-project-results/             (Shared folder - uploaded via API)
└── experiment_results_<timestamp>/
    ├── all_experiment_results.json
    ├── thesis_metrics.csv
    ├── *.png (visualizations)
    └── checkpoints/
```

**Research Traceability:**
- Research Objective: Evaluate deepfake detection under realistic social media conditions
- Methodology: Transfer learning with systematic evaluation across compression, datasets, and architectures
- Implementation: notebooks/experiment_pipeline.ipynb

---
## 1. Environment Setup

In [ ]:
# Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q facenet-pytorch
!pip install -q scikit-learn matplotlib seaborn pandas
!pip install -q pyyaml python-dotenv tqdm
!pip install -q opencv-python-headless
!pip install -q Pillow scikit-image
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("All dependencies installed")

---
## 2. Clone Repository & Mount Drive

In [ ]:
# Clone repository
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/deepfake-social-media-detector")
if not REPO_DIR.exists():
    !git clone https://github.com/nico3783/deepfake-social-media-detector.git /content/deepfake-social-media-detector
else:
    print("Repository already exists")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Mount Google Drive (Datasets account)
from google.colab import drive

drive.mount('/content/drive')

# Update this path to your dataset folder on Drive
DRIVE_DATASET_DIR = Path("/content/drive/MyDrive/deepfake_datasets")

print(f"Dataset directory: {DRIVE_DATASET_DIR}")
if DRIVE_DATASET_DIR.exists():
    print("Contents:")
    for item in sorted(DRIVE_DATASET_DIR.iterdir()):
        print(f"  {item.name}/")

In [ ]:
# Set up Google Drive API for uploading results to shared folder
# This uses the same auth as drive.mount() — no extra login needed

import google.auth
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
creds, _ = google.auth.default()
drive_service = build('drive', 'v3', credentials=creds)

# Shared folder ID (from your link)
RESULTS_FOLDER_ID = "1rHxrAejBGqnBmvmrZsHcjvhM7Cwr7-KJ"

print("Google Drive API ready")
print(f"Results will upload to shared folder: {RESULTS_FOLDER_ID}")

---
## 3. Verify Dataset Structure

In [ ]:
import subprocess

def count_files(d, extensions):
    """Count files with given extensions."""
    count = 0
    for ext in extensions:
        count += len(list(Path(d).rglob(f"*{ext}")))
    return count

def count_videos(d):
    return count_files(d, ['.mp4', '.avi', '.mov', '.mkv', '.webm'])

# Verify FF++ structure
FFPP_DIR = DRIVE_DATASET_DIR / "faceforensic++" / "FF++"
CELEBDF_DIR = DRIVE_DATASET_DIR / "celeb-df"

print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

# FaceForensics++
print(f"\nFaceForensics++:")
if FFPP_DIR.exists():
    print(f"  Path: {FFPP_DIR}")
    for split in ["real", "fake"]:
        split_dir = FFPP_DIR / split
        if split_dir.exists():
            n_vids = count_videos(split_dir)
            print(f"  {split}/: {n_vids} videos")
            # Show first few subdirectories
            subdirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])[:5]
            for sd in subdirs:
                sv = count_videos(sd)
                print(f"    {sd.name}/: {sv} videos")
        else:
            print(f"  {split}/: NOT FOUND")
else:
    print(f"  NOT FOUND at {FFPP_DIR}")

# Celeb-DF
print(f"\nCeleb-DF:")
if CELEBDF_DIR.exists():
    print(f"  Path: {CELEBDF_DIR}")
    for item in sorted(CELEBDF_DIR.iterdir()):
        if item.is_dir():
            n_vids = count_videos(item)
            n_imgs = count_files(item, ['.jpg', '.jpeg', '.png'])
            print(f"  {item.name}/: {n_vids} videos, {n_imgs} images")
    # Check for list_of_testing_videos.txt
    test_list = CELEBDF_DIR / "list_of_testing_videos.txt"
    if test_list.exists():
        with open(test_list) as f:
            n_lines = sum(1 for _ in f)
        print(f"  list_of_testing_videos.txt: {n_lines} videos")
else:
    print(f"  NOT FOUND at {CELEBDF_DIR}")

---
## 4. Frame Extraction & Face Detection

This step extracts frames from raw videos and detects/crops faces using MTCNN.

**Estimated time:** ~30-60 min depending on dataset size and GPU.

In [ ]:
import sys
import cv2
import numpy as np
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm
import csv
import random
import json

# Add repo to path
sys.path.insert(0, str(REPO_DIR))

# Local processing directory (faster than Drive)
LOCAL_DATA_DIR = Path("/content/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = LOCAL_DATA_DIR / "processed"
METADATA_DIR = LOCAL_DATA_DIR / "metadata"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Processing settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FACE_SIZE = 299  # XceptionNet input size
FRAME_INTERVAL = 2  # Extract every Nth frame
MAX_FRAMES_PER_VIDEO = 10  # Max frames per video
CONFIDENCE_THRESHOLD = 0.9  # MTCNN confidence
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
RANDOM_SEED = 42

print(f"Device: {DEVICE}")
print(f"Face size: {FACE_SIZE}x{FACE_SIZE}")
print(f"Frame interval: every {FRAME_INTERVAL} frames")
print(f"Max frames per video: {MAX_FRAMES_PER_VIDEO}")

In [ ]:
# Initialize MTCNN face detector
mtcnn = MTCNN(
    keep_all=False,
    device=DEVICE,
    min_face_size=40,
    thresholds=[0.6, 0.7, 0.7],
    post_process=True,
)
print("MTCNN face detector initialized")

In [ ]:
def extract_faces_from_video(
    video_path: Path,
    output_dir: Path,
    video_id: str,
    label: int,
    dataset_name: str,
    frame_interval: int = 2,
    max_frames: int = 10,
) -> list[dict]:
    """Extract faces from a video file.

    Args:
        video_path: Path to video file.
        output_dir: Directory to save face images.
        video_id: Unique identifier for this video.
        label: 0 for real, 1 for fake.
        dataset_name: Dataset identifier.
        frame_interval: Extract every Nth frame.
        max_frames: Maximum frames per video.

    Returns:
        List of metadata dictionaries.
    """
    metadata_rows = []

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return metadata_rows

    frame_idx = 0
    extracted = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0 and extracted < max_frames:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)

            # Detect face
            try:
                face_tensor = mtcnn(pil_image)
                if face_tensor is not None:
                    # Convert tensor to PIL Image
                    face_np = face_tensor.permute(1, 2, 0).cpu().numpy()
                    face_np = (face_np * 255).astype(np.uint8)
                    face_pil = Image.fromarray(face_np)
                    face_pil = face_pil.resize((FACE_SIZE, FACE_SIZE), Image.Resampling.LANCZOS)

                    # Save face
                    face_filename = f"{video_id}_frame{frame_idx:06d}.jpg"
                    face_path = output_dir / face_filename
                    face_pil.save(face_path, quality=95)

                    metadata_rows.append({
                        "video_path": str(face_path.relative_to(PROCESSED_DIR)),
                        "label": label,
                        "dataset": dataset_name,
                        "video_id": video_id,
                    })
                    extracted += 1
            except Exception as e:
                pass  # Skip frames where face detection fails

        frame_idx += 1

    cap.release()
    return metadata_rows

print("Face extraction function defined")

In [ ]:
# Process FaceForensics++
print("=" * 60)
print("PROCESSING FACEFORENSICS++")
print("=" * 60)

ffpp_metadata = []
ffpp_real_dir = FFPP_DIR / "real"
ffpp_fake_dir = FFPP_DIR / "fake"

# Process real videos
if ffpp_real_dir.exists():
    real_videos = list(ffpp_real_dir.rglob("*.mp4"))
    print(f"\nProcessing {len(real_videos)} real videos...")
    for i, video_path in enumerate(tqdm(real_videos, desc="FF++ real")):
        video_id = f"ffpp_real_{i:04d}"
        face_dir = PROCESSED_DIR / "faces" / "ffpp_real"
        face_dir.mkdir(parents=True, exist_ok=True)

        rows = extract_faces_from_video(
            video_path, face_dir, video_id,
            label=0, dataset_name="faceforensics",
            frame_interval=FRAME_INTERVAL,
            max_frames=MAX_FRAMES_PER_VIDEO,
        )
        ffpp_metadata.extend(rows)

# Process fake videos
if ffpp_fake_dir.exists():
    fake_videos = list(ffpp_fake_dir.rglob("*.mp4"))
    print(f"\nProcessing {len(fake_videos)} fake videos...")
    for i, video_path in enumerate(tqdm(fake_videos, desc="FF++ fake")):
        video_id = f"ffpp_fake_{i:04d}"
        face_dir = PROCESSED_DIR / "faces" / "ffpp_fake"
        face_dir.mkdir(parents=True, exist_ok=True)

        rows = extract_faces_from_video(
            video_path, face_dir, video_id,
            label=1, dataset_name="faceforensics",
            frame_interval=FRAME_INTERVAL,
            max_frames=MAX_FRAMES_PER_VIDEO,
        )
        ffpp_metadata.extend(rows)

print(f"\nFF++ total: {len(ffpp_metadata)} face images extracted")
print(f"  Real: {sum(1 for r in ffpp_metadata if r['label'] == 0)}")
print(f"  Fake: {sum(1 for r in ffpp_metadata if r['label'] == 1)}")

In [ ]:
# Process Celeb-DF
print("=" * 60)
print("PROCESSING CELEB-DF")
print("=" * 60)

celeb_metadata = []

# Process celeb-real videos
celeb_real_dir = CELEBDF_DIR / "Celeb-real"
if celeb_real_dir.exists():
    real_videos = list(celeb_real_dir.rglob("*.mp4"))
    print(f"\nProcessing {len(real_videos)} celeb-real videos...")
    for i, video_path in enumerate(tqdm(real_videos, desc="Celeb real")):
        video_id = f"celeb_real_{i:04d}"
        face_dir = PROCESSED_DIR / "faces" / "celeb_real"
        face_dir.mkdir(parents=True, exist_ok=True)

        rows = extract_faces_from_video(
            video_path, face_dir, video_id,
            label=0, dataset_name="celebdf",
            frame_interval=FRAME_INTERVAL,
            max_frames=MAX_FRAMES_PER_VIDEO,
        )
        celeb_metadata.extend(rows)

# Process celeb-synthesis (fake) videos
celeb_fake_dir = CELEBDF_DIR / "Celeb-synthesis"
if celeb_fake_dir.exists():
    fake_videos = list(celeb_fake_dir.rglob("*.mp4"))
    print(f"\nProcessing {len(fake_videos)} celeb-synthesis videos...")
    for i, video_path in enumerate(tqdm(fake_videos, desc="Celeb fake")):
        video_id = f"celeb_fake_{i:04d}"
        face_dir = PROCESSED_DIR / "faces" / "celeb_fake"
        face_dir.mkdir(parents=True, exist_ok=True)

        rows = extract_faces_from_video(
            video_path, face_dir, video_id,
            label=1, dataset_name="celebdf",
            frame_interval=FRAME_INTERVAL,
            max_frames=MAX_FRAMES_PER_VIDEO,
        )
        celeb_metadata.extend(rows)

# Process youtube-real videos (if needed)
youtube_real_dir = CELEBDF_DIR / "YouTube-real"
if youtube_real_dir.exists():
    yt_videos = list(youtube_real_dir.rglob("*.mp4"))
    print(f"\nProcessing {len(yt_videos)} youtube-real videos...")
    for i, video_path in enumerate(tqdm(yt_videos, desc="YouTube real")):
        video_id = f"yt_real_{i:04d}"
        face_dir = PROCESSED_DIR / "faces" / "yt_real"
        face_dir.mkdir(parents=True, exist_ok=True)

        rows = extract_faces_from_video(
            video_path, face_dir, video_id,
            label=0, dataset_name="celebdf",
            frame_interval=FRAME_INTERVAL,
            max_frames=MAX_FRAMES_PER_VIDEO,
        )
        celeb_metadata.extend(rows)

print(f"\nCeleb-DF total: {len(celeb_metadata)} face images extracted")
print(f"  Real: {sum(1 for r in celeb_metadata if r['label'] == 0)}")
print(f"  Fake: {sum(1 for r in celeb_metadata if r['label'] == 1)}")

---
## 5. Create Train/Val/Test Splits & Metadata CSVs

In [ ]:
# Combine all metadata
all_metadata = ffpp_metadata + celeb_metadata

print(f"Total face images: {len(all_metadata)}")
print(f"  Real: {sum(1 for r in all_metadata if r['label'] == 0)}")
print(f"  Fake: {sum(1 for r in all_metadata if r['label'] == 1)}")

# Group by video_id to keep splits video-level (no data leakage)
from collections import defaultdict
video_groups = defaultdict(list)
for row in all_metadata:
    video_groups[row["video_id"]].append(row)

video_ids = list(video_groups.keys())
random.seed(RANDOM_SEED)
random.shuffle(video_ids)

n_videos = len(video_ids)
n_train = int(n_videos * TRAIN_SPLIT)
n_val = int(n_videos * VAL_SPLIT)

train_videos = video_ids[:n_train]
val_videos = video_ids[n_train:n_train + n_val]
test_videos = video_ids[n_train + n_val:]

print(f"\nVideo-level splits:")
print(f"  Train: {len(train_videos)} videos")
print(f"  Val:   {len(val_videos)} videos")
print(f"  Test:  {len(test_videos)} videos")

# Create split lists
train_rows = [r for vid in train_videos for r in video_groups[vid]]
val_rows = [r for vid in val_videos for r in video_groups[vid]]
test_rows = [r for vid in test_videos for r in video_groups[vid]]

print(f"\nImage-level splits:")
print(f"  Train: {len(train_rows)} images")
print(f"  Val:   {len(val_rows)} images")
print(f"  Test:  {len(test_rows)} images")

In [ ]:
# Save metadata CSVs
def save_metadata_csv(rows, filepath):
    """Save metadata to CSV."""
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["video_path", "label", "dataset", "video_id"])
        writer.writeheader()
        writer.writerows(rows)

save_metadata_csv(train_rows, METADATA_DIR / "train.csv")
save_metadata_csv(val_rows, METADATA_DIR / "val.csv")
save_metadata_csv(test_rows, METADATA_DIR / "test.csv")

# Also save combined metadata
save_metadata_csv(all_metadata, METADATA_DIR / "all_metadata.csv")

# Save split info
split_info = {
    "train_videos": train_videos,
    "val_videos": val_videos,
    "test_videos": test_videos,
    "seed": RANDOM_SEED,
}
with open(METADATA_DIR / "splits.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Metadata saved:")
print(f"  {METADATA_DIR / 'train.csv'}")
print(f"  {METADATA_DIR / 'val.csv'}")
print(f"  {METADATA_DIR / 'test.csv'}")
print(f"  {METADATA_DIR / 'splits.json'}")

---
## 6. Dataset & DataLoader Setup

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image

class DeepfakeDataset(torch.utils.data.Dataset):
    """Dataset for loading pre-extracted face images."""

    def __init__(self, metadata_path, root_dir, mode="train", image_size=299):
        self.root_dir = Path(root_dir)
        self.mode = mode
        self.image_size = image_size

        # Load metadata
        self.samples = []
        with open(metadata_path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append({
                    "path": self.root_dir / row["video_path"],
                    "label": int(row["label"]),
                })

        # Transforms
        if mode == "train":
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(0.5),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["path"]).convert("RGB")
        image = self.transform(image)
        label = torch.tensor(sample["label"], dtype=torch.long)
        return image, label

print("DeepfakeDataset class defined")

In [ ]:
# Create data loaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = DeepfakeDataset(
    metadata_path=METADATA_DIR / "train.csv",
    root_dir=PROCESSED_DIR,
    mode="train",
    image_size=FACE_SIZE,
)

val_dataset = DeepfakeDataset(
    metadata_path=METADATA_DIR / "val.csv",
    root_dir=PROCESSED_DIR,
    mode="val",
    image_size=FACE_SIZE,
)

test_dataset = DeepfakeDataset(
    metadata_path=METADATA_DIR / "test.csv",
    root_dir=PROCESSED_DIR,
    mode="test",
    image_size=FACE_SIZE,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} images, {len(val_loader)} batches")
print(f"Test:  {len(test_dataset)} images, {len(test_loader)} batches")

---
## 7. Model Setup

In [ ]:
from src.models.model_factory import create_model, get_model_summary
from src.config.settings import ModelConfig

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create XceptionNet
xception_config = ModelConfig(
    name="xception",
    num_classes=2,
    pretrained=True,
    dropout_rate=0.5,
)
xception_model = create_model(xception_config, DEVICE)
xception_summary = get_model_summary(xception_model)
print(f"XceptionNet: {xception_summary['total_parameters']:,} parameters")

# Create EfficientNet-B0
efficientnet_config = ModelConfig(
    name="efficientnet",
    num_classes=2,
    pretrained=True,
    dropout_rate=0.2,
)
efficientnet_model = create_model(efficientnet_config, DEVICE)
efficientnet_summary = get_model_summary(efficientnet_model)
print(f"EfficientNet-B0: {efficientnet_summary['total_parameters']:,} parameters")

print(f"\nDevice: {DEVICE}")

---
## 8. Training Functions

In [ ]:
import time
import json
from datetime import datetime

# Output directory
OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def train_model(
    model,
    train_loader,
    val_loader,
    model_name,
    num_epochs=50,
    learning_rate=0.001,
    weight_decay=0.0001,
    label_smoothing=0.1,
    early_stopping_patience=10,
    resume_from_drive=False,
):
    """Train a model with early stopping, checkpointing, and Drive resume."""

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    start_epoch = 0

    # Resume from Drive checkpoint if available
    if resume_from_drive:
        try:
            ckpt_id = load_checkpoint_from_drive(model_name)
            if ckpt_id:
                ckpt_path = Path(f"/content/resume_{model_name}.pt")
                download_checkpoint_from_drive(ckpt_id, ckpt_path)
                ckpt = torch.load(ckpt_path, map_location=DEVICE)
                model.load_state_dict(ckpt["model_state_dict"])
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                start_epoch = ckpt["epoch"] + 1
                print(f"  Resumed from epoch {start_epoch} (val_loss={ckpt['val_loss']:.4f})")
                ckpt_path.unlink(missing_ok=True)
        except Exception as e:
            print(f"  Resume failed, starting fresh: {e}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1, patience=5
    )

    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Training history
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [],
    }

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    checkpoint_dir = OUTPUT_DIR / "checkpoints" / model_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nTraining {model_name} for {num_epochs} epochs (starting from epoch {start_epoch})...")
    start_time = time.time()

    for epoch in range(start_epoch, num_epochs):
        # Train
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = train_correct / train_total

        # Validate
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = loss_fn(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        # Record history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Print progress
        elapsed = time.time() - start_time
        print(
            f"Epoch {epoch+1:3d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
            f"Time: {elapsed/60:.1f}min"
        )

        # Scheduler step
        scheduler.step(val_loss)

        # Early stopping & checkpointing
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            # Save best model
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_acc": val_acc,
            }, checkpoint_dir / "best_model.pt")

            # Backup checkpoint to Drive (survives Colab reload)
            try:
                save_checkpoint_to_drive(
                    model_name, model, epoch, val_loss, val_acc,
                    optimizer.state_dict()
                )
            except Exception as e:
                print(f"  [Drive backup failed: {e}]")
        else:
            patience_counter += 1

        if patience_counter >= early_stopping_patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        model.to(DEVICE)

    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time/60:.1f} minutes")

    # Save history
    history_path = OUTPUT_DIR / f"{model_name}_history.json"
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)
    print(f"History saved to {history_path}")

    return history, model

print("Training function defined")

---
## 8b. Checkpoint helpers (Drive backup & resume)

In [ ]:
import io

def save_checkpoint_to_drive(model_name, model, epoch, val_loss, val_acc, optimizer_state):
    """Save a training checkpoint to Google Drive via API so it survives Colab reloads."""
    drive_folder = get_or_create_subfolder(RESULTS_FOLDER_ID, "checkpoints")
    model_folder = get_or_create_subfolder(drive_folder, model_name)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
        "optimizer_state_dict": optimizer_state,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    # Save to a temp file, then upload
    local_path = Path(f"/content/checkpoint_{model_name}_e{epoch}.pt")
    torch.save(checkpoint, local_path)

    file_metadata = {
        "name": f"checkpoint_e{epoch:03d}.pt",
        "parents": [model_folder],
    }
    media = MediaFileUpload(str(local_path), mimetype="application/octet-stream")
    file = drive_service.files().create(
        body=file_metadata, media_body=media, fields="id"
    ).execute()

    # Also keep best_model.pt updated
    if val_loss < getattr(save_checkpoint_to_drive, f"_best_{model_name}", float("inf")):
        setattr(save_checkpoint_to_drive, f"_best_{model_name}", val_loss)
        # Delete old best if exists
        old_best_id = getattr(save_checkpoint_to_drive, f"_best_id_{model_name}", None)
        if old_best_id:
            try:
                drive_service.files().delete(fileId=old_best_id).execute()
            except Exception:
                pass
        # Upload new best
        file_metadata_best = {
            "name": "best_model.pt",
            "parents": [model_folder],
        }
        file_best = drive_service.files().create(
            body=file_metadata_best, media_body=media, fields="id"
        ).execute()
        setattr(save_checkpoint_to_drive, f"_best_id_{model_name}", file_best["id"])

    local_path.unlink(missing_ok=True)
    print(f"  [Drive] Checkpoint saved: {model_name}/checkpoint_e{epoch:03d}.pt")


def load_checkpoint_from_drive(model_name):
    """Find the latest checkpoint for a model on Drive and return its ID, or None."""
    # List all checkpoints for this model
    query = f"name contains 'checkpoint_e' and trashed = false"
    results = drive_service.files().list(
        q=query, fields="files(id, name, createdTime)",
        orderBy="createdTime desc", pageSize=50
    ).execute()

    model_checkpoints = []
    for f in results.get("files", []):
        if f["name"].startswith("checkpoint_e"):
            model_checkpoints.append(f)

    if not model_checkpoints:
        return None

    # Return the latest
    latest = model_checkpoints[0]
    print(f"  [Drive] Found checkpoint: {latest['name']} ({latest['createdTime']})")
    return latest["id"]


def download_checkpoint_from_drive(file_id, local_path):
    """Download a checkpoint file from Drive to local path."""
    request = drive_service.files().get_media(fileId=file_id)
    with open(local_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
    print(f"  [Drive] Downloaded to {local_path}")


def save_results_to_drive():
    """Upload all local OUTPUT_DIR contents to the shared Drive folder."""
    results_folder = get_or_create_subfolder(RESULTS_FOLDER_ID, "experiment_results_latest")
    count = 0
    for item in OUTPUT_DIR.rglob("*"):
        if item.is_file():
            relative = item.relative_to(OUTPUT_DIR)
            # Create subfolder structure on Drive
            parent_id = results_folder
            if relative.parts[:-1]:
                for part in relative.parts[:-1]:
                    parent_id = get_or_create_subfolder(parent_id, part)
            # Upload file
            file_metadata = {"name": item.name, "parents": [parent_id]}
            media = MediaFileUpload(str(item), mimetype="application/octet-stream")
            drive_service.files().create(
                body=file_metadata, media_body=media, fields="id"
            ).execute()
            count += 1
    print(f"\n[Drive] Uploaded {count} files to shared results folder")


print("Checkpoint helpers defined (Drive backup + resume)")

---
## 9. Train XceptionNet

In [ ]:
# Train XceptionNet (resumes from Drive checkpoint if available)
xception_history, xception_model = train_model(
    model=xception_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name="xception",
    num_epochs=50,
    learning_rate=0.001,
    weight_decay=0.0001,
    label_smoothing=0.1,
    early_stopping_patience=10,
    resume_from_drive=True,
)

---
## 10. Train EfficientNet-B0

In [ ]:
# Train EfficientNet-B0 (resumes from Drive checkpoint if available)
efficientnet_history, efficientnet_model = train_model(
    model=efficientnet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    model_name="efficientnet",
    num_epochs=50,
    learning_rate=0.001,
    weight_decay=0.0001,
    label_smoothing=0.1,
    early_stopping_patience=10,
    resume_from_drive=True,
)

---
## 11. Training Visualization

In [ ]:
import matplotlib.pyplot as plt
import json

def plot_training_curves(history_path, model_name, save_dir):
    """Plot training and validation curves."""
    with open(history_path) as f:
        history = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(history["train_loss"], label="Train", linewidth=2)
    axes[0].plot(history["val_loss"], label="Validation", linewidth=2)
    axes[0].set_title(f"{model_name} - Loss", fontsize=14)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(history["train_acc"], label="Train", linewidth=2)
    axes[1].plot(history["val_acc"], label="Validation", linewidth=2)
    axes[1].set_title(f"{model_name} - Accuracy", fontsize=14)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = save_dir / f"{model_name}_training_curves.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_training_curves(
    OUTPUT_DIR / "xception_history.json",
    "XceptionNet",
    OUTPUT_DIR,
)
plot_training_curves(
    OUTPUT_DIR / "efficientnet_history.json",
    "EfficientNet-B0",
    OUTPUT_DIR,
)

---
## 12. Experiment 1: Compression Impact Quantification

**Contribution C1:** Quantify detection accuracy degradation across c0 (raw), c23 (light), c40 (heavy) compression.

Note: This requires videos saved at different compression levels. If your FF++ videos are already at a single quality, you may need to re-download or re-encode. For now, we evaluate on the test set and note compression level.

In [ ]:
import time

def evaluate_model(model, test_loader, device):
    """Evaluate model on test set."""
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    total_loss = 0.0
    correct = 0
    total = 0

    loss_fn = torch.nn.CrossEntropyLoss()
    start_time = time.time()

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    elapsed = time.time() - start_time

    # Compute metrics
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score, f1_score,
        roc_auc_score, confusion_matrix,
    )

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auroc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    cm = confusion_matrix(all_labels, all_preds)

    return {
        "accuracy": round(accuracy, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "auroc": round(auroc, 4),
        "confusion_matrix": cm.tolist(),
        "total_samples": total,
        "total_time_s": round(elapsed, 4),
        "samples_per_second": round(total / elapsed, 2),
    }

print("Evaluation function defined")

In [ ]:
# Evaluate on test set (all models)
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

results = {}

for model_name, model in [("XceptionNet", xception_model), ("EfficientNet-B0", efficientnet_model)]:
    print(f"\nEvaluating {model_name}...")
    metrics = evaluate_model(model, test_loader, DEVICE)
    results[model_name] = metrics
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  AUROC:     {metrics['auroc']:.4f}")
    print(f"  Speed:     {metrics['samples_per_second']:.1f} samples/s")

# Save results
results_path = OUTPUT_DIR / "test_evaluation.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")

---
## 13. Experiment 4: Practical Deployment Assessment

**Contribution C4:** Measure inference speed, memory usage, and model size.

In [ ]:
import time

def measure_inference_speed(model, input_size, num_iterations=100, batch_size=1, device=None):
    """Measure model inference speed."""
    if device is None:
        device = DEVICE

    model = model.to(device)
    model.eval()

    dummy_input = torch.randn(batch_size, 3, input_size, input_size).to(device)

    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Measure
    start = time.time()
    with torch.no_grad():
        for _ in range(num_iterations):
            _ = model(dummy_input)
    if device.type == "cuda":
        torch.cuda.synchronize()
    end = time.time()

    total_time = end - start
    avg_time = total_time / num_iterations
    fps = batch_size / avg_time

    # Memory
    if device.type == "cuda":
        memory_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    else:
        memory_mb = 0.0

    return {
        "total_time_s": round(total_time, 4),
        "avg_time_per_batch_ms": round(avg_time * 1000, 2),
        "fps": round(fps, 2),
        "memory_mb": round(memory_mb, 2),
    }

print("Speed measurement function defined")

In [ ]:
# Measure inference speed for both models
print("=" * 60)
print("DEPLOYMENT ASSESSMENT")
print("=" * 60)

deployment_results = {}

for model_name, model, input_size in [
    ("XceptionNet", xception_model, 299),
    ("EfficientNet-B0", efficientnet_model, 224),
]:
    print(f"\nProfiling {model_name}...")

    # Model summary
    summary = get_model_summary(model)
    param_count = summary["total_parameters"]
    model_size_mb = param_count * 4 / (1024 ** 2)

    # Inference speed
    speed = measure_inference_speed(model, input_size, num_iterations=100, batch_size=1)

    deployment_results[model_name] = {
        "total_parameters": param_count,
        "trainable_parameters": summary["trainable_parameters"],
        "model_size_mb": round(model_size_mb, 2),
        "input_size": input_size,
        "inference_speed": speed,
    }

    print(f"  Parameters:     {param_count:,}")
    print(f"  Model size:     {model_size_mb:.2f} MB")
    print(f"  Input size:     {input_size}x{input_size}")
    print(f"  FPS:            {speed['fps']:.2f}")
    print(f"  Latency:        {speed['avg_time_per_batch_ms']:.2f} ms")
    print(f"  GPU Memory:     {speed['memory_mb']:.2f} MB")

# Save deployment results
deploy_path = OUTPUT_DIR / "deployment_assessment.json"
with open(deploy_path, "w") as f:
    json.dump(deployment_results, f, indent=2)
print(f"\nDeployment results saved to {deploy_path}")

---
## 14. Confusion Matrix Visualization

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, test_loader, model_name, device, save_dir):
    """Plot confusion matrix."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"])
    plt.title(f"{model_name} - Confusion Matrix", fontsize=14)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    save_path = save_dir / f"{model_name}_confusion_matrix.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_confusion_matrix(xception_model, test_loader, "XceptionNet", DEVICE, OUTPUT_DIR)
plot_confusion_matrix(efficientnet_model, test_loader, "EfficientNet-B0", DEVICE, OUTPUT_DIR)

---
## 15. ROC Curve Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(models_dict, test_loader, device, save_dir):
    """Plot ROC curves for multiple models."""
    plt.figure(figsize=(8, 6))

    for model_name, model in models_dict.items():
        model.eval()
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.numpy())

        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {roc_auc:.4f})")

    plt.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate", fontsize=12)
    plt.ylabel("True Positive Rate", fontsize=12)
    plt.title("ROC Curve Comparison", fontsize=14)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)

    save_path = save_dir / "roc_comparison.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_roc_curves(
    {"XceptionNet": xception_model, "EfficientNet-B0": efficientnet_model},
    test_loader, DEVICE, OUTPUT_DIR,
)

---
## 16. Model Comparison Visualization

In [ ]:
def plot_model_comparison(results, save_dir):
    """Plot model comparison bar chart."""
    metrics = ["accuracy", "precision", "recall", "f1", "auroc"]
    model_names = list(results.keys())

    x = np.arange(len(metrics))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    for i, model_name in enumerate(model_names):
        values = [results[model_name].get(m, 0) for m in metrics]
        bars = ax.bar(x + i * width, values, width, label=model_name)

        # Add value labels on bars
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2., height,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9
            )

    ax.set_ylabel("Score")
    ax.set_title("Model Comparison - Test Set Performance", fontsize=14)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis="y")

    save_path = save_dir / "model_comparison.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_model_comparison(results, OUTPUT_DIR)

---
## 17. Save All Metrics for Chapter 4

In [ ]:
# Combine all experiment results
all_results = {
    "test_evaluation": results,
    "deployment_assessment": deployment_results,
    "training_histories": {
        "xception": str(OUTPUT_DIR / "xception_history.json"),
        "efficientnet": str(OUTPUT_DIR / "efficientnet_history.json"),
    },
    "metadata": {
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "test_samples": len(test_dataset),
        "seed": RANDOM_SEED,
        "device": str(DEVICE),
    },
}

# Save combined results
final_results_path = OUTPUT_DIR / "all_experiment_results.json"
with open(final_results_path, "w") as f:
    json.dump(all_results, f, indent=2)

# Export as CSV for thesis
import csv

csv_path = OUTPUT_DIR / "thesis_metrics.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Model", "Accuracy", "Precision", "Recall", "F1", "AUROC", "FPS", "Latency_ms", "Params", "Size_MB"])
    for model_name in results:
        r = results[model_name]
        d = deployment_results[model_name]
        writer.writerow([
            model_name,
            r["accuracy"], r["precision"], r["recall"], r["f1"], r["auroc"],
            d["inference_speed"]["fps"],
            d["inference_speed"]["avg_time_per_batch_ms"],
            d["total_parameters"],
            d["model_size_mb"],
        ])

print("All results saved:")
print(f"  JSON: {final_results_path}")
print(f"  CSV:  {csv_path}")

---
## 18. Download Results

In [ ]:
# Upload results to shared Google Drive folder via API
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def upload_to_shared_folder(local_dir, folder_id, subfolder_name):
    """Upload a local directory to a Google Drive shared folder."""
    # Create timestamped subfolder in the shared folder
    folder_metadata = {
        'name': subfolder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [folder_id]
    }
    folder = drive_service.files().create(
        body=folder_metadata, fields='id'
    ).execute()
    new_folder_id = folder['id']
    print(f"Created folder: {subfolder_name} (ID: {new_folder_id})")

    # Upload all files recursively
    uploaded = 0
    for item in local_dir.rglob("*"):
        if item.is_file():
            rel_path = str(item.relative_to(local_dir))

            # Create subdirectories as needed
            parts = Path(rel_path).parts
            parent_id = new_folder_id
            for part in parts[:-1]:
                # Check if subdirectory exists
                query = f"name='{part}' and '{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
                results = drive_service.files().list(q=query, fields='files(id)').execute()
                if results['files']:
                    parent_id = results['files'][0]['id']
                else:
                    sub_metadata = {
                        'name': part,
                        'mimeType': 'application/vnd.google-apps.folder',
                        'parents': [parent_id]
                    }
                    sub = drive_service.files().create(
                        body=sub_metadata, fields='id'
                    ).execute()
                    parent_id = sub['id']

            # Upload the file
            file_metadata = {
                'name': item.name,
                'parents': [parent_id]
            }
            media = MediaFileUpload(str(item), resumable=True)
            drive_service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id'
            ).execute()
            uploaded += 1
            print(f"  Uploaded: {rel_path}")

    print(f"\nUploaded {uploaded} files to shared folder")
    return new_folder_id

# Upload all results
subfolder_name = f"experiment_results_{timestamp}"
upload_to_shared_folder(OUTPUT_DIR, RESULTS_FOLDER_ID, subfolder_name)

# Also create downloadable zip as backup
zip_name = f"deepfake_experiment_results_{timestamp}"
!cd /content && zip -r "{zip_name}.zip" outputs/
print(f"\nBackup zip: /content/{zip_name}.zip")

---
## Summary

### What was done:
1. Mounted Google Drive and verified dataset structure
2. Extracted frames from raw videos (FF++ and Celeb-DF)
3. Detected and cropped faces using MTCNN
4. Split data into train/val/test (70/15/15, video-level)
5. Trained XceptionNet and EfficientNet-B0
6. Evaluated on test set (accuracy, precision, recall, F1, AUROC)
7. Measured inference speed, memory, and model size
8. Generated visualizations (training curves, confusion matrices, ROC curves, comparison charts)
9. Exported metrics as JSON and CSV

### Outputs on shared Google Drive folder:
- `experiment_results_<timestamp>/all_experiment_results.json`
- `experiment_results_<timestamp>/thesis_metrics.csv`
- `experiment_results_<timestamp>/*.png` (all visualizations)
- `experiment_results_<timestamp>/checkpoints/` (trained models)

### Next steps:
- Run Experiment 2 (cross-dataset) by retraining on one dataset and testing on the other
- Use `thesis_metrics.csv` for Chapter 4 tables
- Use visualizations for Chapter 4 figures
- Results are uploaded to your shared folder (not the datasets Drive)